# 🛡️ HealthGuard AI — Honest, Reproducible ML Evaluation & Leakage Diagnostics
### Healthcare Software-Defined Networking (SDN) Intrusion Detection System
**Author:** Dishank Jain | **Repository:** [ML-Healthcare-SDN-Cybersecurity](https://github.com/dishankj100/ML-Healthcare-SDN-Cybersecurity)

---

## 🔬 1. Problem Statement: Why 100% Accuracy Was an Illusion

The original `Notebook.ipynb` reported an unrealistic **100.0% accuracy, precision, and recall** across multiple machine learning classifiers. In real-world cybersecurity, empirical 100% detection on complex multi-vector intrusion datasets is virtually impossible.

A rigorous audit revealed **6 critical methodological flaws** that caused massive data leakage and optimistic evaluation bias:
1. **Random Row-Level Splitting on Sequential Switch Snapshots:** OpenFlow counter telemetry was collected at 1-second polling intervals. A random 80/20 train/test split placed consecutive snapshots from the exact same network session into both train and test partitions. The test set was simply memorizing near-identical adjacent frames.
2. **Session Identification vs. Attack Behaviour:** Each class was captured in a distinct individual experiment run. Absolute cumulative byte counters (`port_tx_bytes`, `port_rx_bytes`, `table_active_count`) grew monotonically over the course of each capture run, effectively tagging the specific capture session rather than the invariant behavioural signature of the attack.
3. **Data Leakage in Feature Selection:** `SelectPercentile(mutual_info_classif)` was fit on the **entire dataset** prior to splitting, leaking label-correlation information from the test set into feature selection.
4. **Inverted Metric Calls:** Precision and recall were computed using `precision_score(y_pred, y_test)` instead of `(y_true, y_pred)`.
5. **Arbitrary, Unseeded Class Subsampling:** `.sample(frac=...)` was applied with different unseeded fractions per file (e.g. 0.15 for TCP DDoS, 0.50 for CMD, 0.25 for BruteForce), distorting real class priors.
6. **Flawed Hyperparameter Tuning:** Cross-validation grids used `roc_auc` on multiclass targets (which returned NaN) or evaluated grids across the entire dataset.

### 🎯 Objectives of Notebook v2
- **Pre-flight Chronological Verification:** Mathematically prove that OpenFlow telemetry counters monotonically advance along raw CSV row index.
- **Strict Temporal Splitting:** Split by time within each file (First 70% = Train, 200-row purge buffer, Last ~30% = Test). No pre-shuffling.
- **Fixed 9 SDN Features:** Evaluate the fixed 9 flow-level telemetry features without data-driven leakage.
- **TimeSeriesSplit Tuning:** Tune 8 algorithms strictly on the train partition using forward-chaining cross-validation (`scoring='f1_macro'`).
- **Honest Generalization & Leakage Diagnostics:** Measure duplicate feature rows, quantify feature drift over time, and evaluate performance on unseen normal and attack traffic.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print("Dependencies loaded successfully.")

In [ ]:
ALL_COLUMNS = [
    'src', 'dst', 'table_id', 'ip_bytes', 'ip_packet', 'ip_duration', 'in_port', 'dl_dst', 'port_bytes',
    'port_packet', 'port_flow_count', 'table_active_count', 'table_lookup_count', 'table_matched_count',
    'port_rx_packets', 'port_tx_packets', 'port_rx_bytes', 'port_tx_bytes', 'port_rx_dropped', 'port_tx_dropped',
    'port_rx_errors', 'port_tx_errors', 'port_rx_frame_err', 'port_rx_over_err', 'port_crc_err',
    'port_collisions', 'port_duration_sec'
]

FIXED_9_FEATURES = [
    'ip_bytes', 'ip_packet', 'port_bytes', 'port_packet', 'port_flow_count',
    'table_active_count', 'port_rx_packets', 'port_rx_bytes', 'port_tx_bytes'
]

TRAINING_CLASSES = [
    ('Normal', 'Archieve/normal_ditg.csv'),
    ('BruteForce', 'Archieve/attack_bruteforce.csv'),
    ('CMD', 'Archieve/attack_cmd.csv'),
    ('TCPDDOS', 'Archieve/attack_ddos_tcp.csv'),
    ('UDPDDOS', 'Archieve/attack_ddos_udp.csv'),
    ('Probe', 'Archieve/attack_os_port_scan.csv'),
    ('Smaba', 'Archieve/attack_samba.csv'),
    ('SQLInjection', 'Archieve/attack_sql_injection.csv'),
    ('VNC', 'Archieve/attack_vnc.csv'),
]

CLASS_NAMES = sorted([c[0] for c in TRAINING_CLASSES])
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

print(f"9 Configured Threat Classes (Alphabetical):")
for name in CLASS_NAMES:
    print(f"  Class {CLASS_TO_ID[name]}: {name}")

## ⏱️ 2. Pre-Flight Chronological Verification
Before performing temporal partitioning, we must verify that rows within each capture CSV strictly represent consecutive timestamps.
We inspect `port_duration_sec` (the physical OpenFlow switch port uptime) and `ip_duration` (individual flow entry lifetime).

In [ ]:
chronology_rows = []
for cname, fpath in TRAINING_CLASSES:
    df_check = pd.read_csv(fpath, names=ALL_COLUMNS, usecols=['ip_duration', 'port_duration_sec'], low_memory=False)
    df_check['port_duration_sec'] = pd.to_numeric(df_check['port_duration_sec'], errors='coerce')
    df_check['ip_duration'] = pd.to_numeric(df_check['ip_duration'], errors='coerce')
    df_check = df_check.dropna()
    n = len(df_check)
    
    port_diff = df_check['port_duration_sec'].diff().dropna()
    port_decr = (port_diff < 0).sum()
    port_decr_pct = (port_decr / (n - 1)) * 100 if n > 1 else 0.0
    
    ip_diff = df_check['ip_duration'].diff().dropna()
    ip_decr = (ip_diff < 0).sum()
    ip_decr_pct = (ip_decr / (n - 1)) * 100 if n > 1 else 0.0
    
    chronology_rows.append({
        'File': os.path.basename(fpath),
        'Class': cname,
        'Rows': n,
        'port_duration_decr_%': round(port_decr_pct, 4),
        'ip_duration_decr_%': round(ip_decr_pct, 2),
        'Port_Uptime_Range': f"{df_check['port_duration_sec'].iloc[0]:.0f}s -> {df_check['port_duration_sec'].iloc[-1]:.0f}s",
        'Chronological_Order': 'VERIFIED' if port_decr_pct <= 0.05 else 'NOT_ORDERED'
    })

pd.DataFrame(chronology_rows)

## ✂️ 3. Temporal Splitting with Purge Gap
To prevent lookahead bias and temporal snapshot autocorrelation:
- **First 70% of rows:** Used exclusively for training.
- **200-row purge gap:** 200 rows immediately after the 70% boundary are discarded. This purge gap acts as an embargo to prevent flow table residual states from leaking into the test set.
- **Last ~30% of rows:** Preserved as the held-out temporal test set.
- **Strict Invariant:** Original row order is never shuffled prior to splitting.

In [ ]:
train_chunks = []
test_chunks = []
split_summary = []

for cname, fpath in TRAINING_CLASSES:
    df_raw = pd.read_csv(fpath, names=ALL_COLUMNS, low_memory=False)
    for col in FIXED_9_FEATURES + ['port_duration_sec']:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    df_raw = df_raw.dropna(subset=FIXED_9_FEATURES).reset_index(drop=True)
    df_raw['type'] = CLASS_TO_ID[cname]
    
    n = len(df_raw)
    train_end = int(np.floor(n * 0.70))
    gap_rows = 200
    test_start = train_end + gap_rows
    
    tr = df_raw.iloc[:train_end].copy()
    tr['rel_time'] = np.linspace(0, 1, len(tr))
    train_chunks.append(tr)
    
    te = df_raw.iloc[test_start:].copy()
    test_chunks.append(te)
    
    split_summary.append({
        'Class': cname,
        'Class_ID': CLASS_TO_ID[cname],
        'Total_Rows': n,
        'Train_Rows (70%)': len(tr),
        'Purged_Gap_Rows': test_start - train_end,
        'Test_Rows (~30%)': len(te)
    })

train_df = pd.concat(train_chunks, ignore_index=True)
train_df = train_df.sort_values('rel_time').reset_index(drop=True)
test_df = pd.concat(test_chunks, ignore_index=True)

print(f"Total Train Rows: {len(train_df):,} | Total Test Rows: {len(test_df):,}")
pd.DataFrame(split_summary)

## 🔄 4. TimeSeriesSplit Cross-Validation Strategy
Because capture files were recorded in separate runs, a naive global time split across concatenated classes would leave early cross-validation folds completely missing attack classes.
To solve this, we sort the training set by normalized intra-file relative time progression $t \in [0, 1]$.
When evaluated using `TimeSeriesSplit(n_splits=3)`, each fold evaluates forward-in-time while guaranteeing representation across all 9 classes.

In [ ]:
X_train = train_df[FIXED_9_FEATURES].values
y_train = train_df['type'].values
X_test = test_df[FIXED_9_FEATURES].values
y_test = test_df['type'].values

# Read pre-computed honest metrics from evaluate_honest.py if available
if os.path.exists('results/metrics.csv'):
    metrics_df = pd.read_csv('results/metrics.csv')
    print("Loaded honest evaluation metrics from results/metrics.csv:")
    display(metrics_df[['Model', 'Test_Accuracy', 'Macro_F1', 'Weighted_F1', 'CV_f1_macro', 'Tune_Time_Sec']])
else:
    print("Run `python evaluate_honest.py` to generate complete evaluation metrics.")

## 📊 5. Honest Empirical Results: 8-Model Comparison
Let us examine the true performance of all 8 algorithms on the temporal holdout set.

In [ ]:
if os.path.exists('results/metrics.csv'):
    m_df = pd.read_csv('results/metrics.csv').sort_values('Macro_F1', ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.barplot(data=m_df, x='Macro_F1', y='Model', ax=axes[0], palette='Blues_r')
    axes[0].set_title('Macro F1 Score (Temporal Holdout Test Set)')
    axes[0].set_xlim(0, 1.0)
    
    sns.barplot(data=m_df, x='Test_Accuracy', y='Model', ax=axes[1], palette='Greens_r')
    axes[1].set_title('Test Accuracy (Temporal Holdout Test Set)')
    axes[1].set_xlim(0, 1.0)
    
    plt.tight_layout()
    plt.show()

## 🔍 6. Confusion Matrix & Per-Class Breakdown
In a temporal evaluation, which classes remain detectable, and which classes fail due to counter drift?

In [ ]:
if os.path.exists('results/per_class_report.csv'):
    per_cls = pd.read_csv('results/per_class_report.csv')
    rf_report = per_cls[per_cls['Model'] == 'Random Forest']
    display(rf_report[['Class', 'Precision', 'Recall', 'F1_Score', 'Support']])
    
if os.path.exists('results/confusion_matrix.png'):
    from IPython.display import Image
    display(Image('results/confusion_matrix.png'))

## 🚨 7. Leakage Diagnostic: Random vs. Temporal Split & Duplicate Vectors
Why did the old notebook achieve 100% accuracy?
We compare an unseeded/random 80/20 split against the temporal 70/30 split and count exact duplicate feature vectors shared between train and test.

In [ ]:
if os.path.exists('results/leakage_comparison.csv'):
    leakage_df = pd.read_csv('results/leakage_comparison.csv')
    display(leakage_df)

## 🌐 8. Generalization to Unseen Normal & Attack Traffic
A robust network IDS must distinguish normal clinical traffic even when generated by different applications or protocols.
We evaluate the trained model on:
1. `normal_iperf.csv` (network throughput stress test)
2. `normal_internet1/2/3.csv` (real-world web traffic captures)
3. `ddos_attack_scapy_new.csv` (an unseen DDoS tool)

In [ ]:
if os.path.exists('results/unseen_traffic.csv'):
    display(pd.read_csv('results/unseen_traffic.csv'))
    
if os.path.exists('results/unseen_attack_scapy.csv'):
    display(pd.read_csv('results/unseen_attack_scapy.csv'))

## 📈 9. Feature Drift & Cumulative Counter Invariant Analysis
OpenFlow switches report cumulative counters that monotonically increment over the lifetime of a connection or port (`port_tx_bytes`, `port_rx_bytes`, `port_bytes`).
In this diagnostic, we quantify the extent to which test set feature values exceed the maximum value observed during training.

In [ ]:
if os.path.exists('results/class_drift_summary.csv'):
    display(pd.read_csv('results/class_drift_summary.csv'))

## 🎯 10. Conclusion & Honest Scientific Takeaways
1. **The 100% Accuracy Myth:** The 100% accuracy reported in early work was an artifact of random splitting on autocorrelated time series data, where the test set shared exact and near-duplicate counter values with the training set.
2. **The Reality of Cumulative Counter Drift:** In temporal evaluation, counters continue to grow, causing test instances to occupy feature ranges never seen during training. Tree-based models partition on absolute thresholds, resulting in degraded generalization over extended time horizons.
3. **Implications for Healthcare SDN Security:** Practical SDN security systems must either employ normalized rate features (e.g. bytes-per-second, delta packets) or incorporate active online retraining to maintain high fidelity over changing network states.